# Causal Attention with batching

In [17]:
import torch

inputs = torch.tensor(
    [[0.43, 0.15, 0.89],  # Your      (x^1)
     [0.55, 0.87, 0.66],  # journey   (x^2)
     [0.57, 0.85, 0.64],  # starts    (x^3)
     [0.22, 0.58, 0.33],  # with      (x^4)
     [0.77, 0.25, 0.10],  # one       (x^5)
     [0.05, 0.80, 0.55]]  # step      (x^6)
)

inputs = torch.stack([inputs,inputs],dim=0)
print(inputs.shape)

torch.Size([2, 6, 3])


### The input is batch * num_tokens * embedding_size

In [ ]:
import torch
import torch.nn as nn

class CausalAttentionWithBatching(nn.Module):
    
    def __init__(self,d_in,d_out,context_length,dropout,qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias = qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))
        
        
    def forward(self,x):
        batch_size, num_tokens, embedding_size = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        ## For every batch we need to independently transpose 
        attn_scores = queries @ keys.transpose(1,2)

        ## Inplace Masking
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens,:num_tokens],-torch.inf
        )
        ## :num_tokens is done to ensure that for cases 
        # where the number of tokens in the batch is smaller 
        # than the supported context_size

        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim =-1)

        ## drop out
        attn_weights= self.dropout(attn_weights)

        
        context_vec = attn_weights @ values
        return context_vec

In [19]:
torch.manual_seed(123)

## size of embedding
d_in = inputs.shape[-1]

## context length
context_length = inputs.shape[-2]

## needed final dimension
d_out = 2

ca_batching = CausalAttentionWithBatching(d_in,d_out,context_length,0.5)

output = ca_batching(inputs)
print(output)


tensor([[[-0.9038,  0.4432],
         [-0.4368,  0.2142],
         [-0.4849, -0.1341],
         [-0.5834,  0.0081],
         [-0.6219, -0.0526],
         [-0.1417, -0.0505]],

        [[ 0.0000,  0.0000],
         [-1.1749,  0.0116],
         [-0.7733,  0.0073],
         [-0.9140, -0.2769],
         [-0.7679, -0.0735],
         [-0.6749, -0.0984]]], grad_fn=<UnsafeViewBackward0>)


## use of bufffers 

1. The use of register_buffer in PyTorch is not strictly necessary for all use cases but ofere several advantages here.

2. For instance, wehen we use the CausalAttention calss in outr LLM, buffers are automatically moved to the appropriate device(CPU or GPU) along with out model, which will be relevant when training the LLm in future chapters

3. We dont need to manually ensiure these tensors are on the same device as your model parametres, avoiding device mismatches
